# Description
This notebook takes the two tables data/courses.csv and data/writers.csv as an input and outputs the table time_counts.csv.
It computes for each semester how many writers are mentioned and how many of them are female and male (both in relative in total numbers).

In [6]:
import pandas as pd
import plotly.express as px
from tqdm import tqdm
import numpy as np
import re
import numpy as np
from plotly.subplots import make_subplots
from scipy.stats import linregress

In [24]:
# Read in necessary tables

courses = pd.read_csv("data/courses.csv", sep=";", encoding="utf-8", index_col=0)
writers = pd.read_csv("data/writers.csv", sep=";", encoding="utf-8", index_col=0)

In [25]:
all_semesters = ['wise00-01', 'sose01', 'wise01-02', 'sose02', 'wise02-03', 'sose03', \
                 'wise03-04', 'sose04', 'wise04-05', 'sose05', 'wise05-06', 'sose06', \
                 'wise06-07', 'sose07', 'wise07-08', 'sose08', 'wise08-09', 'sose09',\
                 'wise09-10', 'sose10', 'wise10-11', 'sose11', 'wise11-12', 'sose12', \
                 'wise12-13', 'sose13', 'wise13-14', 'sose14', 'wise14-15', 'sose15', \
                 'wise15-16', 'sose16', 'wise16-17', 'sose17', 'wise17-18', 'sose18',\
                 'wise18-19', 'sose19', 'wise19-20', 'sose20', 'wise20-21', 'sose21', \
                 'wise21-22', 'sose22', 'wise22-23', 'sose23', 'wise23-24']
unis = ["wien", "graz", "basel", "chemnitz", "erfurt", "halle", "marburg", "mainz", "stuttgart"]

In [26]:
# transform gnds back into lists
gnd_ids_lists = []
for ids in courses["NER_GNDs"]:
    if not pd.isna(ids):
        ids = re.sub("\[|\]|\'", "", ids).split(", ")
        gnd_ids_lists.append(ids)
    else:
        gnd_ids_lists.append(np.nan)
courses["NER_GNDs"] = gnd_ids_lists

In [27]:
# Count number of course descriptions per uni
content_counts = {uni: 0 for uni in unis}
for uni in content_counts.keys():
    
    uni_courses = courses[courses["university"] == uni]
    contentcount = len(uni_courses[uni_courses["inhalt_count"]!=0])
    content_counts[uni] = contentcount
#content_counts

In [28]:
data = {}

for uni in unis:
    data[uni+"_total"] = [] # Autoren pro Semester
    data[uni+"_rel"] = [] # Autoren pro Semester, geteilt durch Anzahl Inhalte
    data[uni+"_total_M"] = [] # Anzahl männlicher Autoren
    data[uni+"_total_F"] = [] # Anzahl weiblicher Autoren
    data[uni+"_rel_F"] = [] # Anzahl weiblicher Autoren pro Semester, geteilt durch Anzahl der Autoren
    data[uni+"_rel_M"] = [] # Anzahl männlicher Autoren pro Semester, geteilt durch Anzahl der Autoren
    data[uni+"_relrel_F"] = [] # Anzahl weiblicher Autoren, geteilt durch Anzahl der Autoren, geteilt durch Inhaltsmenge
    data[uni+"_relrel_M"] = [] # Anzahl männlicher Autoren, geteilt durch Anzahl der Autoren, geteilt durch Inhaltsmenge
   

for semester in all_semesters:
    df1 = courses[courses["semester"] == semester]
    for uni in unis:
        df2 = df1[df1["university"] == uni]
        content_count = len(df2[df2["inhalt_count"]!=0])
      
        if content_count != 0:

            all_gnds = []
            for gnds in df2.NER_GNDs:
                if type(gnds) == list:
                    all_gnds.extend(gnds)
    
            # get authors and genders
            authors_genders = []
            for gnd in all_gnds:
                # check if g is an author
                aut = writers[writers["GND-ID"]==gnd].reset_index()
                if len(aut) > 0:
                    gender = aut.reset_index()["gender"].loc[0]
                    authors_genders.append(gender)
    
            anzahl_authors = len(authors_genders)
            #print(anzahl_authors)
            if anzahl_authors != 0:
                anteil_f = authors_genders.count("Weiblich")/anzahl_authors
                anteil_m = authors_genders.count("Männlich")/anzahl_authors
                anzahl_f = authors_genders.count("Weiblich")
                anzahl_m = authors_genders.count("Männlich")
                anteilanteil_f = anteil_f / content_count
                anteilanteil_m = anteil_m / content_count
            else:
                anteil_f = np.nan
                anteil_m = np.nan
                anzahl_f = np.nan
                anzahl_m = np.nan
                anteilanteil_f = np.nan
                anteilanteil_m = np.nan
            anteil_authors = anzahl_authors/content_count
            
        else:
            anzahl_authors = np.nan
            anteil_f = np.nan
            anteil_m = np.nan
            anteil_authors = np.nan
            anzahl_f = np.nan
            anzahl_m = np.nan
            anteilanteil_f = np.nan
            anteilanteil_m = np.nan
            

        data[uni+"_total"].append(anzahl_authors)
        data[uni+"_rel"].append(anteil_authors)
        data[uni+"_rel_F"].append(anteil_f)
        data[uni+"_rel_M"].append(anteil_m)
        data[uni+"_total_F"].append(anzahl_f)
        data[uni+"_total_M"].append(anzahl_m)
        data[uni+"_relrel_F"].append(anteilanteil_f)
        data[uni+"_relrel_M"].append(anteilanteil_m)
        
        
                
            

In [31]:
df = pd.DataFrame(data)
#df["Semester"] = alle_semester
df["semester_start"] = [courses[courses["semester"] == semester].reset_index()["semester_start"].loc[0] for semester in all_semesters]
df['semester_start'] = pd.to_datetime(df['semester_start'])
df.to_csv("data/time_counts.csv", sep=";")